[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [27]:
# ✏️ YOUR IMPLEMENTATION HERE
def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    batch_size, seq_len, d_model = Q.shape

    # initialize output
    O = []

    for i in range(0, seq_len, block_size):
        Q_block = Q[:, i:i + block_size, :]
        # initialize cumsum and max_s
        # Dynamically check how many tokens are actually in this chunk (handles the remainder!)
        q_len = Q_block.shape[1]

        # Initialize using q_len instead of block_size
        cumsum = torch.zeros((batch_size, q_len, 1), device=Q.device)
        max_s = torch.ones((batch_size, q_len, 1), device=Q.device) * (-torch.inf)
        O_block = torch.zeros_like(Q_block)

        for j in range(0, seq_len, block_size):
            K_block = K[:, j: j + block_size, :]
            V_block = V[:, j: j + block_size, :]

            new_S_block = (Q_block @ K_block.transpose(1,2)) / math.sqrt(d_model) # new attention score block
            new_S_max_s = torch.max(new_S_block, dim = -1, keepdim=True).values

            # new max_s
            new_max_s = torch.maximum(max_s, new_S_max_s)
            
            # new cumsum
            block_cusum = torch.sum(torch.exp(new_S_block - new_max_s), dim = -1, keepdim=True)
            new_cumsum = cumsum * torch.exp(max_s - new_max_s) + block_cusum

            # update O
            O_block = (O_block * cumsum * torch.exp(max_s - new_max_s) + torch.exp(new_S_block - new_max_s) @ V_block) / new_cumsum

            # update max_s and cumsum
            max_s = new_max_s
            cumsum = new_cumsum

        O.append(O_block)

    return torch.cat(O, dim=1)


In [28]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [29]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (4.9ms)
  ✅ [2/4] Non-aligned block size (2.1ms)
  ✅ [3/4] Block size invariant (2.4ms)
  ✅ [4/4] Gradient flow (2.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (11.9ms total)
  Progress saved. Run status() to see your dashboard.

